# Neural E-Commerce Search — demo

A quick tour of the two-stage retrieve-and-rank pipeline on Amazon ESCI.
Run the training scripts first (see `docs/training.md`) so that the
`artifacts/` directory contains a trained retriever, reranker, and index.

## 1. Inspect the metrics (no heavy deps required)

The ranking/classification metrics are pure NumPy and illustrate the graded
ESCI relevance scheme.

In [ ]:
import sys; sys.path.insert(0, '../src')
from necs.eval.metrics import ndcg_at_k
from necs.data.preprocess import relevance_gain

# A ranking that puts Exact first, then Substitute, Complement, Irrelevant.
labels = ['E', 'S', 'C', 'I']
gains = [relevance_gain(l) for l in labels]
print('perfect NDCG@10:', round(ndcg_at_k(gains, 10), 4))
print('reversed NDCG@10:', round(ndcg_at_k(gains[::-1], 10), 4))

## 2. Run a live search

Loads the trained artifacts and runs both stages end to end.

In [ ]:
import json
from necs.models.bi_encoder import BiEncoder
from necs.models.cross_encoder import CrossEncoder
from necs.retrieval.index import DenseIndex
from necs.pipeline.search import TwoStageSearcher

catalogue = json.load(open('../artifacts/catalogue.json'))
searcher = TwoStageSearcher(
    bi_encoder=BiEncoder('../artifacts/bi_encoder'),
    cross_encoder=CrossEncoder('../artifacts/cross_encoder'),
    index=DenseIndex.load('../artifacts/product_index'),
    product_text=catalogue,
)

for r in searcher.search('wireless gaming mouse', top_k=5):
    print(f'[{r.label}] {r.score:.3f}  {r.product_text[:70]}')

## 3. Compare against the FastAPI service

With `make serve` running, the same query over HTTP:

In [ ]:
import requests
resp = requests.post(
    'http://localhost:8000/search',
    json={'query': 'wireless gaming mouse', 'top_k': 5},
)
resp.json()